# 🎙️ VoiceBatch Studio v0.0 [Final Upload Fix]
स्पीड फिक्स: वॉयस सैंपल अब बिना अटके सीधे कोलाब में जाएगा।

In [ ]:
# @title 🛠️ Step 1: इंस्टॉलेशन (Fast Mode)
import os
from google.colab import drive
print("⏳ लाइब्रेरीज़ सेटअप हो रही हैं...")
!pip install -q coqpit-config coqui-tts gradio librosa soundfile
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
os.makedirs("outputs", exist_ok=True)
print("✅ तैयार है! अब Step 2 चलाएं।")

In [ ]:
# @title 🚀 Step 2: हाई-स्पीड स्टूडियो लॉन्च करें
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"
tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)

def voice_engine(text, audio_sample, remove_silence):
    if not audio_sample or not text: return None
    parts = re.split(r'(?<=[।?!])\s+', text)
    combined = []
    for p in parts:
        if len(p.strip()) < 2: continue
        wav = tts.tts(text=p, speaker_wav=audio_sample, language='hi')
        combined.append(np.array(wav))
    
    final_wav = np.concatenate(combined)
    if remove_silence: final_wav, _ = librosa.effects.trim(final_wav, top_db=20)
    
    out_path = "outputs/VoiceBatch_Final.wav"
    sf.write(out_path, final_wav, 24000)
    return out_path

with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("# 🎙️ VoiceBatch Studio v0.0")
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label="Script (Hindi)", lines=10)
            counter = gr.Markdown("Shabd: 0")
            txt.change(lambda x: f"Shabd: {len(x.split())}", inputs=[txt], outputs=[counter])
            
            # ⚡ अपलोड को तेज़ करने के लिए 'streaming=True' और 'type=filepath'
            smp = gr.Audio(label="Sample (Turant Upload)", type='filepath', streaming=False)
            
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button("Audio Banayein ⚡", variant="primary")
        with gr.Column():
            out = gr.Audio(label="Download Result")
            
    btn.click(voice_engine, [txt, smp, sil], out)

# ⚡ अपलोड लिमिट और टाइमआउट को फिक्स करने के लिए सेटिंग्स
demo.launch(share=True, debug=True, show_error=True)